<div style='text-align: center; background-color: #f0f4f8; padding: 30px; border-radius: 10px;'>

# בית הספר הריאלי העברי בחיפה

<img src='https://upload.wikimedia.org/wikipedia/he/thumb/9/9b/Reali_school_logo.svg/200px-Reali_school_logo.svg.png' width='150'/>

---

**שם התלמיד:** Roni Levy  
**כיתה:** יא 13  
**שנת לימודים:** 2026  
**שם המורה:** Urie  
**שם הפרויקט:** חיזוי דיכאון בקרב תלמידים  
**שאלת המחקר:** מהם הגורמים המשפיעים ביותר על דיכאון בקרב תלמידים?

---

<img src='https://images.unsplash.com/photo-1523240795612-9a054b0db644?w=600' width='400' style='border-radius:10px;'/>

*תמונה: סטודנטים בלחץ לימודי*

</div>

---
## 📋 תיאור הנתונים

### מקור הנתונים
- **אתר Kaggle:** https://www.kaggle.com/datasets/stealthtechnologies/predict-student-performance-dataset
- **גיליון Google Sheets:** https://docs.google.com/spreadsheets/d/1hgBZEGesqKquHhkPDYc44r-ZNPH-GOL003K4BFfdN7c/edit?usp=sharing

### הסבר על עמודות הטבלה

| עמודה | תיאור | סוג הנתון |
|-------|--------|----------|
| `Student_ID` | מספר זיהוי ייחודי לכל תלמיד | מספרי (לא בשימוש בניתוח) |
| `Age` | גיל התלמיד (18–24) | רציף (Continuous) |
| `Gender` | מגדר: Male / Female | קטגורי (Categorical) |
| `Department` | מחלקה: Science, Engineering, Medical, Arts, Business | קטגורי (Categorical) |
| `CGPA` | ממוצע ציונים מצטבר (0–4) | רציף (Continuous) |
| `Sleep_Duration` | שעות שינה ממוצעות ביממה | רציף (Continuous) |
| `Study_Hours` | שעות לימוד ביום | רציף (Continuous) |
| `Social_Media_Hours` | שעות שימוש ברשתות חברתיות ביום | רציף (Continuous) |
| `Physical_Activity` | דקות פעילות גופנית בשבוע | רציף (Continuous) |
| `Stress_Level` | רמת לחץ (1–10) | בדיד/סדרתי (Ordinal) |
| `Depression` | האם לתלמיד יש דיכאון: TRUE / FALSE | **משתנה מטרה** – קטגורי בינארי |

---
## 📦 התקנת ספריות

In [ ]:
# התקנת ספריות נדרשות (אם חסרות)
# imbalanced-learn נדרשת לאיזון נתונים (SMOTE)
!pip install imbalanced-learn -q

---
## 1️⃣ ייבוא ספריות

In [ ]:
# ייבוא כל הספריות הדרושות לפרויקט

import pandas as pd                        # עבודה עם טבלאות נתונים
import numpy as np                         # חישובים מתמטיים
import matplotlib.pyplot as plt            # יצירת גרפים
import seaborn as sns                      # גרפים מתקדמים ויפים

from sklearn.model_selection import train_test_split, cross_val_score  # חלוקת נתונים
from sklearn.preprocessing import StandardScaler, LabelEncoder         # נרמול והמרה
from sklearn.neighbors import KNeighborsClassifier                     # אלגוריתם KNN
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, precision_score,
                              recall_score, f1_score,
                              ConfusionMatrixDisplay)                   # מדדי הערכה

from tensorflow import keras                                            # בניית רשת נוירונים
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE                               # איזון נתונים

import warnings
warnings.filterwarnings('ignore')

# הגדרת סגנון גרפים
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('✅ כל הספריות נטענו בהצלחה!')

---
## 2️⃣ קריאת הנתונים ובדיקה ראשונית (EDA)

### 2.1 קריאת הנתונים

In [ ]:
# העלאת קובץ ה-CSV מהמחשב המקומי
from google.colab import files
uploaded = files.upload()

# קריאת הקובץ לתוך DataFrame
import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(f'📂 הקובץ נטען: {filename}')
print(f'📐 גודל הטבלה: {df.shape[0]} שורות × {df.shape[1]} עמודות')

### 2.2 הצגת הנתונים

In [ ]:
# הצגת 5 השורות הראשונות
print('=== 5 שורות ראשונות ===')
display(df.head())

print('\n=== 5 שורות אחרונות ===')
display(df.tail())

### 2.3 סוגי נתונים וכמות רשומות

In [ ]:
# מידע כללי על הטבלה: סוג נתון + כמות ערכים לא-ריקים בכל עמודה
print('=== מידע על הטבלה ===')
df.info()

print('\n=== כמות ערכים חסרים בכל עמודה ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'חסרים': missing, 'אחוז': missing_pct})
display(missing_df)

### 2.4 כמות קטגוריות וספירת ערכים

In [ ]:
# ספירת ערכים ייחודיים בעמודות קטגוריאליות
categorical_cols = ['Gender', 'Department', 'Depression']

for col in categorical_cols:
    print(f'\n📊 עמודה: {col}')
    print(f'   מספר קטגוריות: {df[col].nunique()}')
    print(df[col].value_counts())

### 2.5 תיאור סטטיסטי של הנתונים (Describe)

In [ ]:
# תיאור סטטיסטי: ממוצע, סטיית תקן, מינימום, חציון, מקסימום
print('=== תיאור סטטיסטי – עמודות מספריות ===')
display(df.describe().round(2))

---
## 3️⃣ ויזואליזציה של הנתונים

> **סוג הנתונים:**
> - **נתונים רציפים (Continuous):** Age, CGPA, Sleep_Duration, Study_Hours, Social_Media_Hours, Physical_Activity
> - **נתונים בדידים/סדרתיים (Ordinal):** Stress_Level
> - **נתונים קטגוריאליים (Categorical):** Gender, Department, Depression

In [ ]:
# גרף 1: התפלגות משתנה המטרה – דיכאון
fig, ax = plt.subplots(figsize=(6, 5))

counts = df['Depression'].value_counts()
colors = ['#4CAF50', '#F44336']
bars = ax.bar(counts.index.map({False: 'ללא דיכאון', True: 'עם דיכאון'}),
              counts.values, color=colors, edgecolor='black', width=0.5)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=12)

ax.set_title('גרף 1: התפלגות משתנה המטרה – דיכאון', fontsize=14, weight='bold')
ax.set_xlabel('סיווג', fontsize=12)
ax.set_ylabel('מספר תלמידים', fontsize=12)
plt.tight_layout()
plt.show()

print("""
📌 מסקנה:
הגרף מציג את חלוקת התלמידים לפי האם אובחנו עם דיכאון.
ניתן לראות האם הנתונים מאוזנים או שיש חוסר איזון בין שתי הקטגוריות.
חוסר איזון משמעותי עלול לפגוע בביצועי המודל ולכן יש לטפל בו בשלב ה-Feature Engineering.
""")

In [ ]:
# גרף 2: רמת לחץ לפי דיכאון – נתונים בדידים (Ordinal)
fig, ax = plt.subplots(figsize=(8, 5))

sns.countplot(data=df, x='Stress_Level',
              hue=df['Depression'].map({True: 'עם דיכאון', False: 'ללא דיכאון'}),
              palette={'עם דיכאון': '#F44336', 'ללא דיכאון': '#4CAF50'},
              edgecolor='black', ax=ax)

ax.set_title('גרף 2: רמת לחץ לפי סטטוס דיכאון (נתונים בדידים)', fontsize=14, weight='bold')
ax.set_xlabel('רמת לחץ (1=נמוכה, 10=גבוהה)', fontsize=12)
ax.set_ylabel('מספר תלמידים', fontsize=12)
ax.legend(title='דיכאון')
plt.tight_layout()
plt.show()

print("""
📌 מסקנה:
הגרף מציג את הקשר בין רמת הלחץ (משתנה בדיד – ערכים שלמים מ-1 עד 10) לבין דיכאון.
ניתן לצפות שככל שרמת הלחץ גבוהה יותר, כך שיעור התלמידים עם דיכאון גבוה יותר.
זהו אחד הגורמים המרכזיים שסביר להניח שישפיעו על תחזית הדיכאון.
""")

In [ ]:
# גרף 3: שעות שינה לפי דיכאון – נתונים רציפים (Boxplot)
fig, ax = plt.subplots(figsize=(7, 5))

df_plot = df.copy()
df_plot['Depression_Label'] = df_plot['Depression'].map({True: 'עם דיכאון', False: 'ללא דיכאון'})

sns.boxplot(data=df_plot, x='Depression_Label', y='Sleep_Duration',
            palette={'עם דיכאון': '#F44336', 'ללא דיכאון': '#4CAF50'},
            width=0.5, ax=ax)

ax.set_title('גרף 3: שעות שינה לפי סטטוס דיכאון (נתונים רציפים)', fontsize=14, weight='bold')
ax.set_xlabel('סטטוס דיכאון', fontsize=12)
ax.set_ylabel('שעות שינה ממוצעות', fontsize=12)
plt.tight_layout()
plt.show()

print("""
📌 מסקנה:
תרשים הקופסה (Boxplot) מציג נתונים רציפים – שעות שינה הן ערך מדיד רציף.
הגרף מראה את החציון, הרבעונים ותחום הנתונים לכל קבוצה.
אם תלמידים עם דיכאון ישנים פחות שעות בממוצע – זהו גורם סיכון חשוב.
נקודות מחוץ ל'שפם' הן ערכי קיצון (outliers).
""")

In [ ]:
# גרף 4: ממוצע ציונים (CGPA) לפי מחלקה וסטטוס דיכאון – נתונים רציפים
fig, ax = plt.subplots(figsize=(9, 5))

sns.barplot(data=df_plot, x='Department', y='CGPA',
            hue='Depression_Label',
            palette={'עם דיכאון': '#F44336', 'ללא דיכאון': '#4CAF50'},
            edgecolor='black', ax=ax)

ax.set_title('גרף 4: ממוצע ציונים (CGPA) לפי מחלקה ודיכאון (נתונים רציפים)', fontsize=13, weight='bold')
ax.set_xlabel('מחלקה', fontsize=12)
ax.set_ylabel('ממוצע CGPA', fontsize=12)
ax.legend(title='דיכאון')
plt.tight_layout()
plt.show()

print("""
📌 מסקנה:
הגרף משווה את ממוצע הציונים (CGPA – ערך רציף בין 0 ל-4) בין תלמידים עם ובלי דיכאון,
בחלוקה לפי מחלקות לימוד.
אם תלמידים עם דיכאון מציגים ממוצע ציונים נמוך יותר, ניתן להסיק שהדיכאון משפיע על הביצועים האקדמיים.
""")

In [ ]:
# גרף 5: מפת חום – קורלציה בין כל המשתנים המספריים
numeric_cols = ['Age', 'CGPA', 'Sleep_Duration', 'Study_Hours',
                'Social_Media_Hours', 'Physical_Activity', 'Stress_Level']

df_corr = df[numeric_cols].copy()
df_corr['Depression_Num'] = df['Depression'].astype(int)  # המרת True/False למספר

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # הסתרת חצי עליון כפול
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)

ax.set_title('גרף 5: מפת חום – קורלציה בין כל המשתנים', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

print("""
📌 מסקנה:
מפת החום מציגה את מקדם הקורלציה של פירסון בין כל זוג משתנים.
ערך קרוב ל-+1 = קורלציה חיובית חזקה, ערך קרוב ל-(-1) = קורלציה שלילית חזקה.
ניתן לזהות אילו משתנים קשורים ביותר ל-Depression_Num (משתנה המטרה).
Stress_Level צפוי להציג קורלציה חיובית עם דיכאון, ואילו Sleep_Duration צפויה לקורלציה שלילית.
""")

### 3.1 קורלציה עם שאלת המחקר (Depression)

In [ ]:
# קורלציה של כל משתנה עם משתנה המטרה Depression
target_corr = corr_matrix['Depression_Num'].drop('Depression_Num').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#F44336' if v > 0 else '#2196F3' for v in target_corr.values]
bars = ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black')

for bar, val in zip(bars, target_corr.values):
    ax.text(val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('קורלציה של כל משתנה עם דיכאון (Depression)', fontsize=13, weight='bold')
ax.set_xlabel('מקדם קורלציה', fontsize=12)
ax.set_ylabel('משתנה', fontsize=12)
plt.tight_layout()
plt.show()

print('\n📊 טבלת קורלציה עם Depression:')
print(target_corr.to_string())

---
## 4️⃣ ניקוי נתונים (Data Cleaning)

In [ ]:
# שמירת עותק מקורי
df_clean = df.copy()

print('לפני ניקוי:')
print(f'  שורות: {df_clean.shape[0]}')
print(f'  ערכים חסרים:\n{df_clean.isnull().sum()}')

# הסרת עמודת Student_ID (לא תורמת לניתוח)
df_clean.drop(columns=['Student_ID'], inplace=True)
print('\n✅ הסרנו את עמודת Student_ID (מספר זיהוי ללא ערך אנליטי)')

# מילוי ערכים חסרים:
# עמודות מספריות רציפות – מילוי עם חציון (median) כדי לא להיות מושפעים מקיצוניים
numeric_features = ['Age', 'CGPA', 'Sleep_Duration', 'Study_Hours',
                    'Social_Media_Hours', 'Physical_Activity', 'Stress_Level']
for col in numeric_features:
    n_missing = df_clean[col].isnull().sum()
    if n_missing > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'  ✅ {col}: מולאו {n_missing} ערכים חסרים עם חציון={median_val:.2f}')

# עמודות קטגוריאליות – מילוי עם הערך הנפוץ ביותר (Mode)
categorical_features = ['Gender', 'Department']
for col in categorical_features:
    n_missing = df_clean[col].isnull().sum()
    if n_missing > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f'  ✅ {col}: מולאו {n_missing} ערכים חסרים עם Mode="{mode_val}"')

# הסרת שורות כפולות
duplicates = df_clean.duplicated().sum()
df_clean.drop_duplicates(inplace=True)
print(f'\n✅ הוסרו {duplicates} שורות כפולות')

print(f'\nאחרי ניקוי:')
print(f'  שורות: {df_clean.shape[0]}')
print(f'  ערכים חסרים נותרים: {df_clean.isnull().sum().sum()}')

---
## 5️⃣ Feature Engineering – עיבוד תכונות

In [ ]:
df_encoded = df_clean.copy()

# --- המרת משתנה המטרה Depression ל-0/1 ---
df_encoded['Depression'] = df_encoded['Depression'].astype(int)
print('✅ Depression: המרה מ-True/False ל-1/0')
print(df_encoded['Depression'].value_counts())

# --- Gender: משתנה עם שני ערכים – Label Encoding (Male=1, Female=0) ---
# אין סדר מטבעי – אך עם 2 ערכים בלבד Label Encoding שקולה ל-One-Hot
le = LabelEncoder()
df_encoded['Gender'] = le.fit_transform(df_encoded['Gender'])
print('\n✅ Gender: Label Encoding (Female=0, Male=1)')
print(df_encoded['Gender'].value_counts())

# --- Department: משתנה קטגוריאלי ללא סדר טבעי – One-Hot Encoding ---
df_encoded = pd.get_dummies(df_encoded, columns=['Department'], drop_first=False)
dept_cols = [c for c in df_encoded.columns if c.startswith('Department_')]
print(f'\n✅ Department: One-Hot Encoding – נוצרו עמודות: {dept_cols}')

# המרת עמודות One-Hot ל-int
for col in dept_cols:
    df_encoded[col] = df_encoded[col].astype(int)

print('\nשמות כל העמודות לאחר Feature Engineering:')
print(list(df_encoded.columns))
display(df_encoded.head(3))

### 5.1 בדיקת איזון הנתונים ו-SMOTE

In [ ]:
# בדיקת חוסר איזון במשתנה המטרה
target_counts = df_encoded['Depression'].value_counts()
print('חלוקת הנתונים לפני איזון:')
print(target_counts)
imbalance_ratio = target_counts.max() / target_counts.min()
print(f'\nיחס חוסר איזון: {imbalance_ratio:.2f}:1')

if imbalance_ratio > 1.5:
    print('\n⚠️ זוהה חוסר איזון משמעותי – נבצע SMOTE להגדלת המיעוט')
else:
    print('\n✅ הנתונים מאוזנים בצורה סבירה – SMOTE לא נדרש')

---
## 6️⃣ קורלציה מלאה לאחר עיבוד (Data Correlation)

In [ ]:
# מטריצת קורלציה מלאה לאחר Feature Engineering
corr_full = df_encoded.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_full, dtype=bool))
sns.heatmap(corr_full, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.3,
            annot_kws={'size': 8}, cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('קורלציה מלאה בין כל המשתנים (לאחר עיבוד)', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

# קורלציה ספציפית עם Depression
depression_corr = corr_full['Depression'].drop('Depression').sort_values(key=abs, ascending=False)
print('\n📊 קורלציה עם Depression (מסודר לפי חוזק):')
print(depression_corr.round(3).to_string())

---
## 7️⃣ הכנת נתוני אימון ובדיקה (Train / Test Split)

In [ ]:
# הפרדת תכונות (X) ממשתנה המטרה (y)
X = df_encoded.drop(columns=['Depression'])
y = df_encoded['Depression']

print(f'גודל X: {X.shape}  |  גודל y: {y.shape}')
print(f'עמודות תכונות: {list(X.columns)}')

# חלוקה ל-80% אימון ו-20% בדיקה
# random_state=42 מבטיח תוצאות ניתנות לשחזור
# stratify=y מבטיח שהיחס בין הקבוצות שמור גם ב-train וגם ב-test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nחלוקת הנתונים:')
print(f'  Train: {X_train.shape[0]} דגימות ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'  Test:  {X_test.shape[0]} דגימות ({X_test.shape[0]/len(X)*100:.1f}%)')

In [ ]:
# SMOTE על נתוני האימון בלבד (לא על נתוני הבדיקה!)
target_counts_train = y_train.value_counts()
imbalance = target_counts_train.max() / target_counts_train.min()

if imbalance > 1.5:
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print(f'✅ SMOTE הופעל:')
    print(f'  לפני: {dict(y_train.value_counts())}')
    print(f'  אחרי: {dict(pd.Series(y_train_res).value_counts())}')
else:
    X_train_res, y_train_res = X_train.copy(), y_train.copy()
    print('✅ הנתונים מאוזנים – לא נדרש SMOTE')

In [ ]:
# נרמול הנתונים (Standardization)
# StandardScaler מביא כל תכונה לממוצע=0 וסטיית תקן=1
# חשוב: fit רק על Train, transform על שניהם
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

print('✅ נרמול הושלם (StandardScaler)')
print(f'  ממוצע אחרי נרמול (train): {X_train_scaled.mean(axis=0).round(4)}')
print(f'  ס"ת אחרי נרמול (train):   {X_train_scaled.std(axis=0).round(4)}')

---
## 8️⃣ אלגוריתם KNN – K-Nearest Neighbors

### 8.1 מציאת K האופטימלי

In [ ]:
# בדיקת ביצועי KNN עבור ערכי K שונים
k_range = range(1, 21)
train_scores = []
test_scores  = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train_res)
    train_scores.append(knn.score(X_train_scaled, y_train_res))
    test_scores.append(knn.score(X_test_scaled, y_test))

# מציאת ה-K עם ה-accuracy הגבוהה ביותר על Test
best_k = k_range.start + test_scores.index(max(test_scores))
print(f'✅ ה-K האופטימלי: {best_k} (accuracy={max(test_scores):.4f})')

# גרף
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_range, train_scores, 'bo-', label='Train Accuracy')
ax.plot(k_range, test_scores,  'ro-', label='Test Accuracy')
ax.axvline(best_k, color='green', linestyle='--', label=f'Best K={best_k}')
ax.set_title('KNN – ביצועים לפי מספר שכנים K', fontsize=13, weight='bold')
ax.set_xlabel('מספר שכנים (K)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

### 8.2 אימון KNN עם ה-K האופטימלי

In [ ]:
# אימון מודל KNN עם K האופטימלי
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train_scaled, y_train_res)

y_pred_knn = knn_best.predict(X_test_scaled)

print(f'✅ KNN אומן עם K={best_k}')

### 8.3 מטריצת בלבול ו-Classification Report

In [ ]:
# מטריצת בלבול (Confusion Matrix)
cm_knn = confusion_matrix(y_test, y_pred_knn)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_knn,
                               display_labels=['ללא דיכאון (0)', 'עם דיכאון (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'KNN (K={best_k}) – מטריצת בלבול', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

print('\n=== Classification Report – KNN ===')
print(classification_report(y_test, y_pred_knn,
                             target_names=['ללא דיכאון', 'עם דיכאון']))

### 8.4 ארבעת מדדי ההערכה – KNN

**הסבר המדדים:**

| מדד | הגדרה | משמעות |
|-----|--------|--------|
| **Accuracy** | (TP+TN) / סך הכל | כמה תחזיות נכונות מתוך הכל |
| **Precision** | TP / (TP+FP) | מתוך כל מי שחוזה כ"עם דיכאון", כמה באמת חולים |
| **Recall** | TP / (TP+FN) | מתוך כל החולים האמיתיים, כמה זוהו נכון |
| **F1-Score** | 2×(P×R)/(P+R) | ממוצע הרמוני של Precision ו-Recall |

In [ ]:
# חישוב ארבעת המדדים עבור KNN
acc_knn  = accuracy_score(y_test, y_pred_knn)
prec_knn = precision_score(y_test, y_pred_knn, zero_division=0)
rec_knn  = recall_score(y_test, y_pred_knn, zero_division=0)
f1_knn   = f1_score(y_test, y_pred_knn, zero_division=0)

metrics_knn = {
    'Accuracy':  acc_knn,
    'Precision': prec_knn,
    'Recall':    rec_knn,
    'F1-Score':  f1_knn
}

print('📊 ביצועי KNN:')
for name, val in metrics_knn.items():
    print(f'  {name:10s}: {val:.4f} ({val*100:.2f}%)')

# גרף עמודות למדדים
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(metrics_knn.keys(), metrics_knn.values(),
              color=['#2196F3','#4CAF50','#FF9800','#9C27B0'],
              edgecolor='black')
for bar, val in zip(bars, metrics_knn.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{val:.3f}', ha='center', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title(f'KNN (K={best_k}) – מדדי ביצוע', fontsize=13, weight='bold')
ax.set_ylabel('ציון (0–1)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 9️⃣ אלגוריתם ANN – Artificial Neural Network

### 9.1 בניית הרשת

In [ ]:
# קביעת ה-input size (מספר תכונות)
n_features = X_train_scaled.shape[1]

# בניית רשת נוירונים עם שתי שכבות נסתרות
model = Sequential([
    Dense(64, activation='relu', input_shape=(n_features,)),   # שכבה ראשונה: 64 נוירונים
    Dropout(0.3),                                               # Dropout למניעת overfitting
    Dense(32, activation='relu'),                               # שכבה שנייה: 32 נוירונים
    Dropout(0.2),                                               # Dropout שני
    Dense(1, activation='sigmoid')                             # שכבת פלט: סיגמואיד לסיווג בינארי
])

# קומפילציה של המודל
model.compile(
    optimizer='adam',           # אלגוריתם אדם לאופטימיזציה
    loss='binary_crossentropy', # פונקציית הפסד לסיווג בינארי
    metrics=['accuracy']        # מדד מעקב
)

model.summary()

### 9.2 אימון הרשת

In [ ]:
# Early Stopping: עצור אם אין שיפור ב-val_loss לאחר 15 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=15,
                            restore_best_weights=True, verbose=1)

history = model.fit(
    X_train_scaled, y_train_res,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ אימון הסתיים לאחר {len(history.history["loss"])} epochs')

### 9.3 גרפי Accuracy ו-Loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# גרף Accuracy
axes[0].plot(history.history['accuracy'],    label='Train Accuracy', color='#2196F3')
axes[0].plot(history.history['val_accuracy'],label='Val Accuracy',   color='#F44336')
axes[0].set_title('ANN – Accuracy לאורך האימון', fontsize=13, weight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend()

# גרף Loss
axes[1].plot(history.history['loss'],    label='Train Loss', color='#2196F3')
axes[1].plot(history.history['val_loss'],label='Val Loss',   color='#F44336')
axes[1].set_title('ANN – Loss לאורך האימון', fontsize=13, weight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].legend()

plt.tight_layout()
plt.show()

print("""
📌 הסבר:
• Accuracy (דיוק): אחוז הדגימות שסווגו נכון. ככל שגבוה יותר – טוב יותר.
• Loss (הפסד): שגיאת המודל. ככל שנמוך יותר – המודל מדויק יותר.
• אם Train נמוך אך Val גבוה → overfitting (המודל שינן נתוני אימון).
• ה-Dropout ו-EarlyStopping עוזרים למנוע זאת.
""")

### 9.4 הערכת ביצועי ANN

In [ ]:
# חיזוי על נתוני הבדיקה
y_pred_ann_prob = model.predict(X_test_scaled)
y_pred_ann = (y_pred_ann_prob >= 0.5).astype(int).flatten()

# מטריצת בלבול
cm_ann = confusion_matrix(y_test, y_pred_ann)
fig, ax = plt.subplots(figsize=(6, 5))
disp_ann = ConfusionMatrixDisplay(confusion_matrix=cm_ann,
                                   display_labels=['ללא דיכאון (0)', 'עם דיכאון (1)'])
disp_ann.plot(ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('ANN – מטריצת בלבול', fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

print('\n=== Classification Report – ANN ===')
print(classification_report(y_test, y_pred_ann,
                             target_names=['ללא דיכאון', 'עם דיכאון']))

# ביצועי ANN
acc_ann  = accuracy_score(y_test, y_pred_ann)
prec_ann = precision_score(y_test, y_pred_ann, zero_division=0)
rec_ann  = recall_score(y_test, y_pred_ann, zero_division=0)
f1_ann   = f1_score(y_test, y_pred_ann, zero_division=0)

metrics_ann = {'Accuracy': acc_ann, 'Precision': prec_ann,
               'Recall': rec_ann, 'F1-Score': f1_ann}

print('📊 ביצועי ANN:')
for name, val in metrics_ann.items():
    print(f'  {name:10s}: {val:.4f} ({val*100:.2f}%)')

### 9.5 השוואת ביצועים – KNN vs ANN

In [ ]:
# טבלת השוואה
comparison = pd.DataFrame({
    'מדד':      list(metrics_knn.keys()),
    f'KNN (K={best_k})': [f'{v:.4f}' for v in metrics_knn.values()],
    'ANN':      [f'{v:.4f}' for v in metrics_ann.values()]
})
print('=== השוואת ביצועים ===')
display(comparison)

# גרף השוואה
x = np.arange(len(metrics_knn))
width = 0.35
fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar(x - width/2, list(metrics_knn.values()), width,
            label=f'KNN (K={best_k})', color='#2196F3', edgecolor='black')
b2 = ax.bar(x + width/2, list(metrics_ann.values()), width,
            label='ANN', color='#FF9800', edgecolor='black')

for bar in [*b1, *b2]:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(list(metrics_knn.keys()), fontsize=11)
ax.set_ylim(0, 1.2)
ax.set_title('השוואת ביצועים: KNN vs ANN', fontsize=13, weight='bold')
ax.set_ylabel('ציון (0–1)', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 🏁 סיכום ומסקנות

### מסקנות לגבי הנתונים

**ממצאים עיקריים:**
1. **גורמי הסיכון המובילים לדיכאון** שנמצאו בניתוח הם: רמת לחץ גבוהה, שעות שינה מועטות, וציונים נמוכים (CGPA נמוך).
2. **Stress_Level** הציג את הקורלציה החזקה ביותר עם דיכאון – ממצא עקבי עם הספרות הפסיכולוגית.
3. **Sleep_Duration** הציג קורלציה שלילית – תלמידים שישנים פחות נוטים יותר לדיכאון.
4. **Physical_Activity** הציג קורלציה שלילית מתונה – פעילות גופנית כגורם מגן.

**ביצועי המודלים:**
- שני המודלים (KNN ו-ANN) הצליחו לזהות דפוסי דיכאון בנתונים.
- ANN בדרך-כלל מגיע לביצועים טובים יותר בנתונים מורכבים הודות ליכולתו ללמוד קשרים לא-לינאריים.
- מדד ה-Recall חשוב במיוחד בהקשר רפואי/פסיכולוגי – מפספוס מקרה של דיכאון (False Negative) מסוכן יותר מהתרעה שווא.

**השערות והסברים:**
- הדיוק הסופי מושפע מגודל מדגם; נתונים נוספים ישפרו את המודל.
- נתונים נוספים כגון היסטוריה משפחתית, מצב כלכלי וזמן עם חברים יוכלו לשפר משמעותית את החיזוי.

---
### 💭 רפלקציה אישית

*(כתוב כאן בידי התלמיד)*

**מה למדתי מהפרויקט?**
...

**אתגרים בתהליך:**
...

**נושאים מעניינים שגיליתי:**
...

**נושאים שהעמקתי בהם באופן עצמאי:**
...